# Model Training

## Objectives
- Prepare image datasets for training
- Build a Convolutional Neural Network (CNN)
- Train the model to classify cherry leaves
- Apply techniques to prevent overfitting

## Inputs
- Training and validation datasets

## Outputs
- Trained CNN model
- Training history (accuracy and loss)
- Saved model file

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

In [ ]:
import splitfolders

input_folder = "../inputs/cherry-leaves"
output_folder = "../outputs/datasets"

splitfolders.ratio(
    input_folder,
    output=output_folder,
    seed=42,
    ratio=(.7, .2, .1)
)

In [ ]:
train_path = "../outputs/datasets/train"
val_path = "../outputs/datasets/val"
test_path = "../outputs/datasets/test"

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
IMG_SIZE = (100, 100)
BATCH_SIZE = 20

In [ ]:
train_data = train_datagen.flow_from_directory(
    train_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

val_data = val_datagen.flow_from_directory(
    val_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

In [ ]:
model = models.Sequential([
    
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(100,100,3)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),  # prevent overfitting
    
    layers.Dense(1, activation='sigmoid')
])

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    callbacks=[early_stop]
)

In [ ]:
model.save("../outputs/models/mildew_detector.h5")

In [ ]:
import pandas as pd

history_df = pd.DataFrame(history.history)
history_df.to_csv("../outputs/models/training_history.csv", index=False)